In [1]:
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestClassifier

class MRIFusionEngine:
    def __init__(self):
        self.save_dir = "deployment/fusion_assets"
        os.makedirs(self.save_dir, exist_ok=True)
        self.classes = ['ACL Tear', 'Meniscus Tear', 'General Abnormality']
        self.model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

    def train_model(self):
        print("⚙️ Training MRI Clinical Reasoner...")
        samples = 2000
        X_volume = np.random.rand(samples, len(self.classes))
        X_clinical = np.column_stack((np.random.randint(18, 90, samples), np.random.randint(0, 2, samples), np.random.randint(0, 2, samples)))
        X = np.hstack((X_volume, X_clinical))
        y = np.argmax(X_volume, axis=1)
        self.model.fit(X, y)
        joblib.dump(self.model, os.path.join(self.save_dir, "mri_fusion.joblib"))
        print("✅ MRI Fusion Model Saved.")

    def generate_report(self, volume_probs, age, gender, prev_diag):
        top_idx = np.argmax(volume_probs)
        condition = self.classes[top_idx]
        base_conf = volume_probs[top_idx]

        multiplier = 1.0
        reasoning = []
        reasoning.append(f"1. 2.5D Volumetric Analysis: Spatial feature stacking via the VGG16 backbone identified hyperintense signal anomalies consistent with {condition} (Baseline AI Confidence: {base_conf*100:.1f}%).")

        if age > 50 and condition == 'Meniscus Tear':
            multiplier += 0.15
            reasoning.append(f"2. Demographic Correlation: Patient age ({age}) strongly suggests a degenerative meniscal fraying etiology rather than an acute, high-impact ligamentous trauma.")
        elif age <= 35 and condition == 'ACL Tear':
            multiplier += 0.12
            reasoning.append(f"3. Demographic Correlation: Patient age ({age}) aligns perfectly with high-impact/sports-related acute anterior cruciate ligament rupture profiles.")

        if prev_diag:
            multiplier += 0.20
            reasoning.append(f"4. Longitudinal Risk Factor: Previous ipsilateral knee trauma or surgical intervention significantly increases the biomechanical risk of secondary structural failures.")

        final_conf = min(base_conf * multiplier, 0.99)
        reasoning.append(f"► Final Assessment (Weighted Conf: {final_conf*100:.1f}%): Structural compromise verified for {condition}. Recommend orthopedic evaluation for potential arthroscopic intervention.")

        return {"diagnosis": condition, "confidence": round(final_conf, 4), "reasoning": reasoning}

if __name__ == "__main__":
    MRIFusionEngine().train_model()

⚙️ Training MRI Clinical Reasoner...
✅ MRI Fusion Model Saved.
